In [ ]:
import pandas as pd
import mysql.connector
import os
from dotenv import load_dotenv
import pycountry
from fuzzywuzzy import process
from geopy.geocoders import Nominatim
import pandas as pd
import time


## basic data overview/cleaning

In [ ]:


customers=pd.read_csv('data/raw/customers.csv')
geolocation=pd.read_csv('data/raw/geolocation.csv')
order_items=pd.read_csv('data/raw/order_items.csv')
orders=pd.read_csv('data/raw/orders.csv')
payments=pd.read_csv('data/raw/payments.csv')
products=pd.read_csv('data/raw/products.csv')
sellers=pd.read_csv('data/raw/sellers.csv')


In [ ]:
customers.info()
customers.isnull().sum()
customers.duplicated().sum()
customers['customer_city'].unique()
# changing the city names to title case
customers['customer_city']=customers['customer_city'].apply(lambda x: x.title())
customers.head()
os.makedirs('data/cleaned', exist_ok=True)
customers.to_csv('data/cleaned/customers_cleaned.csv', index=False)


In [ ]:
geolocation.isnull().sum()
geolocation.info()
geolocation.head(20)
geolocation['geolocation_cleaned_city'].unique()
# clean city names
geolocation['geolocation_cleaned_city']=geolocation['geolocation_city'].apply(lambda x: str(x).strip().replace('$','S').replace('£', '').replace('@','').title())
geolocation=geolocation.drop(columns='geolocation_city')
geolocation.head()
geolocation.to_csv('data/cleaned/geolocation_cleaned.csv', index=False)

In [ ]:
payments.describe()
payments.info()
payments.isnull().sum()
payments.head()
payments.to_csv('data/cleaned/payments_cleaned.csv', index=False)


In [ ]:
sellers.describe()
sellers.info()
sellers.isnull().sum()
sellers.head()
sellers['seller_city'].unique()
sellers['seller_city']=sellers['seller_city'].apply(lambda x: str(x).strip().title())
sellers.to_csv('data/cleaned/sellers_cleaned.csv', index=False)


In [ ]:
order_items.head()
order_items.info()
order_items.isnull().sum()
order_items['shipping_limit_date']=pd.to_datetime(order_items['shipping_limit_date'])
order_items.to_csv('data/cleaned/order_items_cleaned.csv', index=False)


In [ ]:
orders.describe().T #for better readability
orders.isnull().sum()
orders.info()
orders.dtypes
# date conversions
from datetime import datetime
orders['order_purchase_timestamp']=pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at']=pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date']=pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date']=pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date']=pd.to_datetime(orders['order_estimated_delivery_date'])

orders.to_csv('data/cleaned/orders_cleaned.csv', index=False)



In the Orders dataset, the null values are present in columns where they can be ignored and the analysis can still take place without removing them, so later we will only change them to <i>None</i> for SQL analysis.

In [ ]:
products.describe()
products=products.dropna() # dropped the null values since they were present in important columns
products.isnull().sum()
products.to_csv('data/cleaned/products_cleaned.csv', index=False)


## connecting and loading data to mysql server

In [ ]:
load_dotenv()
db_pass = os.getenv("PASSWORD")

# List of CSV files and their corresponding table names
csv_files = [
    ('customers_cleaned.csv', 'customers'),
    ('geolocation_cleaned.csv', 'geolocation'),
    ('order_items_cleaned.csv', 'order_items'),
    ('orders_cleaned.csv', 'orders'),
    ('payments_cleaned.csv', 'payments'),
    ('products_cleaned.csv', 'products'),
    ('sellers_cleaned.csv', 'sellers'),
]

# Connect to the MySQL database
conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password=db_pass,
        database='ecommerce_sales'
    )
cursor = conn.cursor()


# Folder containing the CSV files
folder_path = 'C:/Users/Lenovo/Desktop/My_Notebooks/e-commerce-sales-project/data/cleaned'

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'TEXT'

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
    
    # Replace NaN with None to handle SQL NULL
    df = df.where(pd.notnull(df), None)



    # Debugging: Check for NaN values
    print(f"Processing {csv_file}")
    # print(f"NaN values before replacement:\n{df.isnull().sum()}\n")

    # Clean column names
    df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') for col in df.columns]

    # Generate the CREATE TABLE statement with appropriate data types
    columns = ', '.join([f'`{col}` {get_sql_type(df[col].dtype)}' for col in df.columns])
    create_table_query = f'CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})'
    cursor.execute(create_table_query)

    # Insert DataFrame data into the MySQL table
    for _, row in df.iterrows():
        # Convert row to tuple and handle NaN/None explicitly
        values = tuple(None if pd.isna(x) else x for x in row)
        sql = f"INSERT INTO `{table_name}` ({', '.join(['`' + col + '`' for col in df.columns])}) VALUES ({', '.join(['%s'] * len(row))})"
        cursor.execute(sql, values)

    # Commit the transaction for the current CSV file
    conn.commit()

# Close the connection
conn.close()